# B3 — data-driven label-count sweep (when do labels make PI worthwhile?)

**Closing the last open piece of Professor Rabczuk's item 4**: the existing PI-vs-DD comparison has exactly ONE data-driven data point (N=200 samples: 2.51% error, ~2.21h total cost, vs. PI's 1.88% at ~2.12h, no labels needed) -- not the break-even analysis his 2026-10-20 email actually asked for ("figure out when the expensive high-fidelity labels actually make PI worthwhile").

**This notebook adds two more points, bracketing N=200** (scope decided with Omar -- a 2-point bracket, not a full 5-point sweep, to keep this to ~4 GPU-hours of new training instead of ~8):

- **N=50**: reuses the first 50 rows of the EXISTING 200-sample training set (`b3_dd_training_set`) -- no new FEM solves, since these are already-generated i.i.d. samples from the same distribution. Label cost for N=50 is reported as `50 * 4.13s/sample` (the measured per-sample rate), i.e. what it would have cost to generate exactly 50 samples, not what this notebook actually spends (it spends ~0, reusing existing data).
- **N=400**: the existing 200 samples PLUS 200 NEWLY generated samples at a disjoint seed (`seed=51`, sample_seed range 510,000-510,199 -- clearly disjoint from the existing training range 500,000-500,199 and the held-out range 999990-1000089).

Both trained with `--loss rel_l2` (the better-performing plain DD variant, avoiding conflating this sweep with a loss-type comparison) at the SAME 50,000-step budget as every other B3 run, and evaluated against the SAME held-out set -- so all four points (PI, DD@50, DD@200, DD@400) are directly, apples-to-apples comparable. Each run also reports the H1 semi-norm (added 2026-10-26).

**What this answers**: does DD's accuracy improve enough with more labels to close the gap with PI (1.88%), and does the total cost (labels + training) ever become competitive with PI's label-free 2.12h -- or does the already-measured N=200 point (already slightly worse AND slightly more expensive than PI) mean DD simply doesn't catch up within a practical sample range for B3? Either answer is the genuine finding Professor Rabczuk asked for.

In [ ]:
# =====================================================================
#  CELL -- B3 data-driven label-count sweep: N=50 (reused subset, no new
#  FEM solves) and N=400 (existing 200 + 200 newly generated), both
#  trained at the standard 50,000-step budget and evaluated against the
#  same held-out set as every other B3 checkpoint.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import h5py
import numpy as np
import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()

R = '/content/drive/MyDrive/pfem_run'
HOLDOUT_DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
EXISTING_200 = f'{R}/b3_dd_training_set/dataset.h5'
assert os.path.exists(EXISTING_200), f'expected the existing 200-sample DD training set at {EXISTING_200}'

PER_SAMPLE_LABEL_COST_S = 4.13  # measured 2026-10-24 when the original 200-sample set was generated

# ---------------------------------------------------------------------
# N=50: a SUBSET of the existing 200-sample dataset, no new FEM solves.
# ---------------------------------------------------------------------
print(f'\n{"=" * 90}')
print('=== Building N=50 dataset (subset of the existing 200, no new FEM solves) ===')
N50_DIR = f'{R}/b3_dd_training_set_n50'
os.makedirs(N50_DIR, exist_ok=True)
N50_DATASET = f'{N50_DIR}/dataset.h5'
with h5py.File(EXISTING_200, 'r') as src, h5py.File(N50_DATASET, 'w') as dst:
    for key in src.keys():
        dst.create_dataset(key, data=src[key][:50])
print(f'Wrote {N50_DATASET} (50 samples, reused from the existing training set).')

# ---------------------------------------------------------------------
# N=400: existing 200 + 200 NEWLY generated samples at a disjoint seed.
# ---------------------------------------------------------------------
print(f'\n{"=" * 90}')
print('=== Generating 200 NEW samples (seed=51, disjoint from the existing seed=50 range) ===')
from omar_pfem.data.data_generate_B3_dataset import generate_dataset

EXTRA200_DIR = f'{R}/b3_dd_training_set_extra200'
os.makedirs(EXTRA200_DIR, exist_ok=True)
device_str = 'cuda'

t0 = time.time()
generate_dataset(num_samples=200, output_dir=EXTRA200_DIR, seed=51, device=device_str)
gen_elapsed_new200 = time.time() - t0
print(f'Generated 200 new samples in {gen_elapsed_new200:.1f}s '
      f'({gen_elapsed_new200 / 200:.2f}s/sample).')

print(f'\n{"=" * 90}')
print('=== Concatenating existing 200 + new 200 into an N=400 dataset ===')
N400_DIR = f'{R}/b3_dd_training_set_n400'
os.makedirs(N400_DIR, exist_ok=True)
N400_DATASET = f'{N400_DIR}/dataset.h5'
with h5py.File(EXISTING_200, 'r') as a, h5py.File(f'{EXTRA200_DIR}/dataset.h5', 'r') as b, \
     h5py.File(N400_DATASET, 'w') as dst:
    for key in a.keys():
        dst.create_dataset(key, data=np.concatenate([a[key][:], b[key][:]], axis=0))
print(f'Wrote {N400_DATASET} (400 samples).')

# ---------------------------------------------------------------------
# Train both N=50 and N=400 at the standard 50,000-step budget.
# ---------------------------------------------------------------------
SWEEP_POINTS = [
    (50, N50_DATASET, 0.0),  # label cost reported separately (see summary below)
    (400, N400_DATASET, gen_elapsed_new200),
]

results = []
for N, dataset_path, new_label_gen_s in SWEEP_POINTS:
    print(f'\n{"=" * 90}')
    print(f'=== Training data-driven B3 operator, N={N}, --loss rel_l2 ===')
    OUT_DIR = f'{R}/b3_data_driven_rel_l2_N{N}'
    os.makedirs(OUT_DIR, exist_ok=True)
    sys.argv = [
        'train_B3_data_driven.py',
        '--train_dataset', dataset_path,
        '--test_dataset', HOLDOUT_DATASET,
        '--out_dir', OUT_DIR,
        '--loss', 'rel_l2',
        '--opt_steps', '50000',
        '--batch_size', '8',
        '--eval_every', '2000',
    ]
    t0 = time.time()
    from omar_pfem.train_B3_data_driven import main as train_dd_main
    train_dd_main()
    train_elapsed = time.time() - t0

    with open(f'{OUT_DIR}/data_driven_B3_rel_l2.json') as f:
        report = json.load(f)

    full_label_cost_s = N * PER_SAMPLE_LABEL_COST_S
    total_cost_s = full_label_cost_s + train_elapsed
    results.append({
        'N': N, 'best_val_rel_L2': report['best_val_rel_L2'],
        'best_val_H1_seminorm_rel': report.get('best_val_H1_seminorm_rel'),
        'train_wall_clock_s': train_elapsed,
        'label_cost_s_(at_measured_4.13s/sample_rate)': full_label_cost_s,
        'total_cost_s': total_cost_s, 'total_cost_h': total_cost_s / 3600,
    })

    try:
        from omar_pfem.run_manifest import write_manifest
        write_manifest(OUT_DIR, kind='b3_data_driven_breakeven_sweep',
                        args={'N': N, 'loss': 'rel_l2', 'opt_steps': 50000},
                        started_at=_started,
                        results=results[-1],
                        outputs=[f'{OUT_DIR}/model_best.pt', f'{OUT_DIR}/data_driven_B3_rel_l2.json'],
                        notes="Break-even sweep point (Professor Rabczuk's 2026-10-20 email: "
                              "\'figure out when the expensive high-fidelity labels actually "
                              "make PI worthwhile\'), bracketing the existing N=200 result.")
    except Exception as e:
        print(f'[manifest] not recorded: {e}')

print(f'\n{"=" * 90}')
print('=== Break-even summary: PI vs. DD at N=50, 200 (existing), 400 ===')
print('  PI baseline            : 1.88% combined displacement error, ~2.12h training, 0 label cost')
print('  DD rel_l2 @ N=200 (existing): 2.51% rel. L2, ~2.21h total (0.23h labels + 1.98h training)')
for r in results:
    print(f'  DD rel_l2 @ N={r["N"]:<4}: {r["best_val_rel_L2"]*100:.2f}% rel. L2, '
          f'H1_seminorm={r["best_val_H1_seminorm_rel"]:.4f}, '
          f'{r["total_cost_h"]:.2f}h total ('
          f'{r["label_cost_s_(at_measured_4.13s/sample_rate)"]/3600:.2f}h labels + '
          f'{r["train_wall_clock_s"]/3600:.2f}h training)')

print('\nDone. Send this cell\'s full printed output back -- especially the break-even summary '
      'table above -- to see whether DD ever catches up to PI within this sample range.')
